In [170]:
## 1. ASK
## How many days last season had optimal conditions for snowmaking?
## Estimate the electricity budget required to run all snowmaking equipment for every hour with optimal conditions.
## Graph all days of the season using a heatmap.
## Color all days with a gradient according to the conditions for snowmaking. Use facets to separate each month.
## Graph all snow that fell naturally last season using a line.

## 2a. PREPARE / RESEARCH
## The winter season is November 1, 2024 - March 31, 2025
## Assuming trails open at 9AM and close at 4:30PM
## Snowmaking crews work for 16.5 hours
## To make snow, temperature must be 27 degrees Farenheit and below
## Humidity must be low between 30% and 60%
## Snowshoe mountain is equipped with 75 DemacLenko Titan 2.0 fixed position tower fan guns.
## 90 low-energy "stick" guns which use 10% of the energy.

In [171]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [172]:
import sys
print(sys.version)
print('pandas version', pd.__version__)

3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:37:03) [MSC v.1929 64 bit (AMD64)]
pandas version 2.2.3


In [173]:
## This dataset provided by the NCEI or National Centers for Environmental Information has a record of daily snowfall totals.
## As provided, this table has a title. I'm going to include a parameter header=1 to begin the index on the second row.

In [174]:
sn = pd.read_csv("C:/Users/Spencer Woo/Downloads/data(9).csv", header=1)

In [175]:
sn

,Date,TAVG (Degrees Fahrenheit),TMAX (Degrees Fahrenheit),TMIN (Degrees Fahrenheit),PRCP (Inches),SNOW (Inches),SNWD (Inches)
0,1975-02-01,NaN,40.0,31.0,0.46,0.0,0.0
1,1975-02-02,NaN,34.0,27.0,0.40,4.0,4.0
2,1975-02-03,NaN,36.0,19.0,0.00,0.0,NaN
3,1975-02-04,NaN,30.0,14.0,0.00,0.0,NaN
4,1975-02-05,NaN,45.0,17.0,1.18,15.0,17.0
...,...,...,...,...,...,...,...
18571,2025-12-06,NaN,27.0,18.0,0.01,0.5,3.0
18572,2025-12-07,NaN,30.0,17.0,0.00,0.0,NaN
18573,2025-12-08,NaN,35.0,19.0,0.01,0.2,3.0
18574,2025-12-09,NaN,26.0,14.0,0.30,2.0,5.0


In [176]:
sn.columns

Index(['Date', 'TAVG (Degrees Fahrenheit)', 'TMAX (Degrees Fahrenheit)',
       'TMIN (Degrees Fahrenheit)', 'PRCP (Inches)', 'SNOW (Inches)',
       'SNWD (Inches)'],
      dtype='object')

In [177]:
sn[['Date', 'SNOW (Inches)']]

,Date,SNOW (Inches)
0,1975-02-01,0.0
1,1975-02-02,4.0
2,1975-02-03,0.0
3,1975-02-04,0.0
4,1975-02-05,15.0
...,...,...
18571,2025-12-06,0.5
18572,2025-12-07,0.0
18573,2025-12-08,0.2
18574,2025-12-09,2.0


In [178]:
sn_ss = sn[['Date', 'SNOW (Inches)']]

In [179]:
## This dataset is 18,576 rows long
## I only need the '24-'25 winter season as a subset
## Start of season is November 11
## End of season is March 30

In [180]:
sn_ss.query("Date >= '2024-11-01' and Date <= '2025-04-30'")

,Date,SNOW (Inches)
18171,2024-11-01,0.0
18172,2024-11-02,0.0
18173,2024-11-03,0.0
18174,2024-11-04,0.0
18175,2024-11-05,0.0
...,...,...
18347,2025-04-26,0.0
18348,2025-04-27,0.0
18349,2025-04-28,0.0
18350,2025-04-29,0.0


In [181]:
sn_ss = sn_ss.query("Date >= '2024-11-01' and Date <= '2025-04-30'")

In [182]:
## 17 days are missing daily snowfall

In [183]:
sn_ss.isna().sum()

Date              0
SNOW (Inches)    17
dtype: int64

In [184]:
sn_ss.query('`SNOW (Inches)`.isna()')

,Date,SNOW (Inches)
18209,2024-12-09,NaN
18210,2024-12-10,NaN
18211,2024-12-11,NaN
18213,2024-12-13,NaN
18215,2024-12-15,NaN
18217,2024-12-17,NaN
18218,2024-12-18,NaN
18219,2024-12-19,NaN
18225,2024-12-25,NaN
18226,2024-12-26,NaN


In [185]:
sn_ss = ss_ss.fillna(0)

In [186]:
sn_ss.isna().sum()

Date             0
SNOW (Inches)    0
dtype: int64

In [ ]:
## There was no snow on Christmas!

In [187]:
sn_ss.loc[sn_ss['Date'] == '2024-12-25']

,Date,SNOW (Inches)
18225,2024-12-25,0.0


In [188]:
## This is a dataset from WeatherUnderground, they keep historical weather data from the timeframe we need.
## The reason I chose this dataset is the humidity column.
## The data is also very precise, with observations being roughly an hour apart.

In [58]:
el = pd.read_csv("C:/Users/Spencer Woo/Downloads/Elkins-Randolph County Regional Airport Station - Sheet1.csv")

In [59]:
el.shape

(5230, 11)

In [60]:
el.head()

,Date,Time,Temperature,Dew Point,Humidity,Wind,Wind Speed,Wind Gust,Pressure,Precip.,Condition
0,11-01-2024,11:51 PM,68 °F,51 °F,55 %,S,7 mph,0 mph,27.98 in,0.0 in,Cloudy
1,11-01-2024,12:51 AM,67 °F,52 °F,59 %,S,9 mph,0 mph,27.98 in,0.0 in,Cloudy
2,11-01-2024,1:51 AM,67 °F,53 °F,61 %,S,6 mph,0 mph,27.97 in,0.0 in,Cloudy
3,11-01-2024,2:51 AM,66 °F,56 °F,70 %,W,7 mph,0 mph,27.98 in,0.0 in,Cloudy
4,11-01-2024,3:51 AM,65 °F,57 °F,75 %,WSW,6 mph,0 mph,27.99 in,0.0 in,Light Rain


In [61]:
el.dtypes

Date           object
Time           object
Temperature    object
Dew Point      object
Humidity       object
Wind           object
Wind Speed     object
Wind Gust      object
Pressure       object
Precip.        object
Condition      object
dtype: object

In [62]:
el.columns

Index(['Date', 'Time', 'Temperature', 'Dew Point', 'Humidity', 'Wind',
       'Wind Speed', 'Wind Gust', 'Pressure', 'Precip.', 'Condition'],
      dtype='object')

In [63]:
## Copy and paste the results of el.columns into double brackets to take a subset of the data

In [64]:
el[['Date', 'Time', 'Temperature', 'Humidity']]

,Date,Time,Temperature,Humidity
0,11-01-2024,11:51 PM,68 °F,55 %
1,11-01-2024,12:51 AM,67 °F,59 %
2,11-01-2024,1:51 AM,67 °F,61 %
3,11-01-2024,2:51 AM,66 °F,70 %
4,11-01-2024,3:51 AM,65 °F,75 %
...,...,...,...,...
5225,03-31-2025,8:51 PM,49 °F,97 %
5226,03-31-2025,9:01 PM,49 °F,97 %
5227,03-31-2025,9:30 PM,47 °F,93 %
5228,03-31-2025,9:51 PM,46 °F,93 %


In [65]:
el = el[['Date', 'Time', 'Temperature', 'Humidity']]

In [66]:
## Remember, the conditions for making snow are a temperature of 27 degrees farenheit and below Humidity must be between 30 and 60 percent
## This is every observation of the season where snowmaking was possible

In [67]:
el.query("Temperature <= '27' and Humidity >= '30' and Humidity <= '60'")

,Date,Time,Temperature,Humidity
1114,11-30-2024,5:51 PM,24 °F,57 %
1115,11-30-2024,6:51 PM,24 °F,55 %
1169,12-01-2024,9:51 PM,22 °F,58 %
1337,12-07-2024,9:51 AM,25 °F,55 %
1543,12-12-2024,6:51 AM,26 °F,58 %
...,...,...,...,...
4420,03-02-2025,9:51 AM,22 °F,53 %
4421,03-02-2025,10:51 AM,25 °F,46 %
4430,03-02-2025,7:51 PM,25 °F,51 %
4431,03-02-2025,8:51 PM,24 °F,55 %


In [68]:
## Save this query as a subset of the original dataset without modifying the original

In [69]:
el_ss = el.query("Temperature <= '27' and Humidity >= '30' and Humidity <= '60'")

In [70]:
el_ss.dtypes

Date           object
Time           object
Temperature    object
Humidity       object
dtype: object

In [71]:
el_ss['Datetime_Col'] = el_ss['Date'] + ' ' + el_ss['Time']

C:\Users\Spencer Woo\AppData\Local\Temp\ipykernel_7500\1164856074.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  el_ss['Datetime_Col'] = el_ss['Date'] + ' ' + el_ss['Time']


In [72]:
el_ss

,Date,Time,Temperature,Humidity,Datetime_Col
1114,11-30-2024,5:51 PM,24 °F,57 %,11-30-2024 5:51 PM
1115,11-30-2024,6:51 PM,24 °F,55 %,11-30-2024 6:51 PM
1169,12-01-2024,9:51 PM,22 °F,58 %,12-01-2024 9:51 PM
1337,12-07-2024,9:51 AM,25 °F,55 %,12-07-2024 9:51 AM
1543,12-12-2024,6:51 AM,26 °F,58 %,12-12-2024 6:51 AM
...,...,...,...,...,...
4420,03-02-2025,9:51 AM,22 °F,53 %,03-02-2025 9:51 AM
4421,03-02-2025,10:51 AM,25 °F,46 %,03-02-2025 10:51 AM
4430,03-02-2025,7:51 PM,25 °F,51 %,03-02-2025 7:51 PM
4431,03-02-2025,8:51 PM,24 °F,55 %,03-02-2025 8:51 PM


In [73]:
el_ss['Datetime_Col'] = pd.to_datetime(el_ss['Datetime_Col'])

C:\Users\Spencer Woo\AppData\Local\Temp\ipykernel_7500\1442229862.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  el_ss['Datetime_Col'] = pd.to_datetime(el_ss['Datetime_Col'])
C:\Users\Spencer Woo\AppData\Local\Temp\ipykernel_7500\1442229862.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  el_ss['Datetime_Col'] = pd.to_datetime(el_ss['Datetime_Col'])


In [74]:
el_ss.dtypes

Date                    object
Time                    object
Temperature             object
Humidity                object
Datetime_Col    datetime64[ns]
dtype: object

In [75]:
el_ss.sort_values('Datetime_Col')

,Date,Time,Temperature,Humidity,Datetime_Col
1114,11-30-2024,5:51 PM,24 °F,57 %,2024-11-30 17:51:00
1115,11-30-2024,6:51 PM,24 °F,55 %,2024-11-30 18:51:00
1169,12-01-2024,9:51 PM,22 °F,58 %,2024-12-01 21:51:00
1337,12-07-2024,9:51 AM,25 °F,55 %,2024-12-07 09:51:00
1543,12-12-2024,6:51 AM,26 °F,58 %,2024-12-12 06:51:00
...,...,...,...,...,...
4060,12-18-2025,5:51 PM,21 °F,55 %,2025-12-18 17:51:00
4061,12-18-2025,6:51 PM,20 °F,57 %,2025-12-18 18:51:00
4062,12-18-2025,7:51 PM,19 °F,54 %,2025-12-18 19:51:00
4063,12-18-2025,8:51 PM,17 °F,59 %,2025-12-18 20:51:00


In [76]:

el_ss['hours_since_last'] = el_ss.groupby(el_ss['Datetime_Col'].dt.date)['Datetime_Col'].diff().fillna(pd.Timedelta(seconds=0))

C:\Users\Spencer Woo\AppData\Local\Temp\ipykernel_7500\477777036.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  el_ss['hours_since_last'] = el_ss.groupby(el_ss['Datetime_Col'].dt.date)['Datetime_Col'].diff().fillna(pd.Timedelta(seconds=0))


In [77]:
el_ss['hours_since_last'] = el_ss['hours_since_last'].dt.total_seconds() / 3600

C:\Users\Spencer Woo\AppData\Local\Temp\ipykernel_7500\3552716958.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  el_ss['hours_since_last'] = el_ss['hours_since_last'].dt.total_seconds() / 3600


In [78]:
el_ss

,Date,Time,Temperature,Humidity,Datetime_Col,hours_since_last
1114,11-30-2024,5:51 PM,24 °F,57 %,2024-11-30 17:51:00,0.0
1115,11-30-2024,6:51 PM,24 °F,55 %,2024-11-30 18:51:00,1.0
1169,12-01-2024,9:51 PM,22 °F,58 %,2024-12-01 21:51:00,0.0
1337,12-07-2024,9:51 AM,25 °F,55 %,2024-12-07 09:51:00,0.0
1543,12-12-2024,6:51 AM,26 °F,58 %,2024-12-12 06:51:00,0.0
...,...,...,...,...,...,...
4420,03-02-2025,9:51 AM,22 °F,53 %,2025-03-02 09:51:00,1.0
4421,03-02-2025,10:51 AM,25 °F,46 %,2025-03-02 10:51:00,1.0
4430,03-02-2025,7:51 PM,25 °F,51 %,2025-03-02 19:51:00,9.0
4431,03-02-2025,8:51 PM,24 °F,55 %,2025-03-02 20:51:00,1.0


In [79]:
el_ss['hours_since_last'].sum()

np.float64(74.0)

In [80]:
## There were 74 hours in the season where conditions were optimal for snowmaking
## I'm saving this value as a variable for the electricity budget estimate

In [81]:
op_hr = el_ss['hours_since_last'].sum()

In [82]:
## 2b. PREPARE / RESEARCH
## DemacLenko Titan 2.0 power consumption: 14-24 kW
## SnowLogic DV7
## In all, Snowshoe boasts 15 compressors
## Ingersroll Rand power consumption: 895 kW
## 10 pumps
## Average power consumption for a turbine pump: 930 kW
## Mon Power Commercial Rates: General service (Schedule B) includes an energy charge of $0.12808 per kWh for the first 400 kWh
## and $0.11403 for additional usage

In [83]:
def sum_kw():
    total_sum = 0
    data = [
        [24, 75],
        [24 * .10, 90],
        [930, 10],
        [895, 15]
    ]
    for row in data:
        product = row[0] * row[1]
        total_sum += product
    return total_sum

def convert_kwh():
    return sum_kw() * op_hr

def estimate_budget(ra = 0.12808, kwha = 400, rb = 0.11403):
    return f"${kwha * ra + (convert_kwh() - kwha) * rb:,.2f}"

estimate_budget()

'$208,775.62'

In [198]:
data = {
    'Tool': ['DemacLenko Titan 2.0 Fan Gun', 'SnowLogic DV7 Stick Gun', 'Turbine Pump', 'Ingersoll Rand Compressor'],
    'Kilowatts': [24, 24 * .10, 930, 895],
    'Quantity': [75, 90, 10, 15]
}

In [199]:
ni = pd.DataFrame(data)

In [200]:
ni

,Tool,Kilowatts,Quantity
0,DemacLenko Titan 2.0 Fan Gun,24.0,75
1,SnowLogic DV7 Stick Gun,2.4,90
2,Turbine Pump,930.0,10
3,Ingersoll Rand Compressor,895.0,15


In [203]:
ni['Product'] = ni['Kilowatts'] * ni['Quantity']

In [204]:
ni

,Tool,Kilowatts,Quantity,Product
0,DemacLenko Titan 2.0 Fan Gun,24.0,75,1800.0
1,SnowLogic DV7 Stick Gun,2.4,90,216.0
2,Turbine Pump,930.0,10,9300.0
3,Ingersoll Rand Compressor,895.0,15,13425.0


In [206]:
ni['Product'].sum()

np.float64(24741.0)

In [207]:
sum_kw = ni['Product'].sum()

In [208]:
def convert_kwh():
    return sum_kw * op_hr

In [209]:
convert_kwh()

np.float64(1830834.0)

In [210]:
def estimate_budget(ra = 0.12808, kwha = 400, rb = 0.11403):
    return f"${kwha * ra + (convert_kwh() - kwha) * rb:,.2f}"

In [211]:
estimate_budget()

'$208,775.62'